# Chapter 12: Capstone Project: Cleaning a Messy Sales Dataset

Companion notebook for *Master Data Cleaning: A Hands-On Beginner's Guide with Python, pandas & NumPy*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 12.2 Loading and Diagnosing

In [1]:
import pandas as pd

orders_raw = pd.read_csv('orders.csv')
orders_raw.shape

(200, 7)

In [2]:
orders_raw.dtypes

order_id       str
order_date     str
customer_id    str
product        str
city           str
quantity       str
unit_price     str
dtype: object

In [3]:
orders_raw['quantity'].value_counts()

quantity
1       54
4       42
5       34
2       33
3       32
9999     2
-        1
Name: count, dtype: int64

In [4]:
orders_raw.isna().sum()

order_id       0
order_date     3
customer_id    0
product        0
city           6
quantity       2
unit_price     4
dtype: int64

In [5]:
orders_raw['city'].value_counts()

city
New York          58
Los Angeles       26
Philadelphia      24
San Francisco     17
Seattle           11
Denver             9
 new york          8
NYC                5
NEW YORK           4
 Denver            4
San Francisco      4
san francisco      3
philadelphia       3
LA                 3
SEATTLE            2
new york           2
-                  2
Philly             2
PHILADELPHIA       2
los angeles        2
Los Angeles        1
seattle            1
denver             1
Name: count, dtype: int64

In [6]:
orders_raw.duplicated().sum()

np.int64(4)

## 12.3 Fixing Data Types

In [7]:
orders = pd.read_csv('orders.csv', na_values=['-'])
orders.dtypes

order_id           str
order_date         str
customer_id        str
product            str
city               str
quantity       float64
unit_price         str
dtype: object

In [8]:
price_text = (
    orders['unit_price']
    .str.replace('$', '')
    .str.replace(',', '')
    .str.strip()
)
price_text.head()

0     259.00
1     129.00
2        259
3      19.90
4    1099.00
Name: unit_price, dtype: str

In [9]:
missing_before = orders['unit_price'].isna().sum()
orders['unit_price'] = pd.to_numeric(price_text, errors='coerce')
missing_after = orders['unit_price'].isna().sum()
print(missing_before, missing_after)

5 5


In [10]:
orders.isna().sum()

order_id       0
order_date     3
customer_id    0
product        0
city           8
quantity       3
unit_price     5
dtype: int64

## 12.4 Handling Missing Values

In [11]:
orders = orders.dropna(subset=['quantity', 'order_date'])
orders.shape

(194, 7)

In [12]:
product_median = (
    orders.groupby('product')['unit_price'].transform('median')
)
orders['unit_price'] = orders['unit_price'].fillna(product_median)
orders['city'] = orders['city'].fillna('Unknown')
orders.isna().sum()

order_id       0
order_date     0
customer_id    0
product        0
city           0
quantity       0
unit_price     0
dtype: int64

## 12.5 Removing Duplicates

In [13]:
print(orders.duplicated().sum())
print(orders.duplicated(subset=['order_id']).sum())

4
6


In [14]:
repeated = orders.duplicated(subset=['order_id'], keep=False)
orders.loc[repeated, ['order_id', 'customer_id', 'city', 'quantity']]

,order_id,customer_id,city,quantity
18,ORD-1019,C002,Philadelphia,4.0
20,ORD-1019,C002,Philadelphia,4.0
52,ORD-1052,C037,New York,3.0
54,ORD-1052,C037,New York,3.0
69,ORD-1068,C040,New York,1.0
73,ORD-1068,C040,New York,1.0
79,ORD-1077,C005,New York,4.0
80,ORD-1077,C005,new york,4.0
92,ORD-1089,C049,Los Angeles,3.0
95,ORD-1089,C049,Los Angeles,3.0


In [15]:
orders = orders.drop_duplicates(subset=['order_id'], keep='first')
orders.shape

(188, 7)

## 12.6 Standardizing Text

In [16]:
orders['city'] = orders['city'].str.strip().str.title()
orders['city'].value_counts()

city
New York         65
Los Angeles      27
Philadelphia     27
San Francisco    24
Seattle          14
Denver           14
Unknown           8
Nyc               5
Philly            2
La                2
Name: count, dtype: int64

In [17]:
city_names = {
    'Nyc': 'New York',
    'La': 'Los Angeles',
    'Philly': 'Philadelphia',
}
orders['city'] = orders['city'].replace(city_names)
orders['city'].value_counts()

city
New York         70
Los Angeles      29
Philadelphia     29
San Francisco    24
Seattle          14
Denver           14
Unknown           8
Name: count, dtype: int64

In [18]:
fixed_id = orders['customer_id'].str.strip().str.upper()
changed = orders['customer_id'] != fixed_id
orders.loc[changed, 'customer_id'].tolist()

['c042', ' C016', 'c039', ' C035', 'c044', ' C037']

In [19]:
orders['customer_id'] = fixed_id
orders['customer_id'].str.len().value_counts()

customer_id
4    188
Name: count, dtype: int64

## 12.7 Parsing Mixed Date Formats

In [20]:
date_formats = ['%Y-%m-%d', '%d/%m/%Y', '%d %b %Y']

parsed = pd.to_datetime(orders['order_date'], format=date_formats[0],
                        errors='coerce')
for fmt in date_formats[1:]:
    attempt = pd.to_datetime(orders['order_date'], format=fmt,
                             errors='coerce')
    parsed = parsed.fillna(attempt)

parsed.isna().sum()

np.int64(0)

In [21]:
print(parsed.min(), parsed.max())

2025-01-01 00:00:00 2025-06-30 00:00:00


In [22]:
orders['order_date'] = parsed
orders['order_month'] = orders['order_date'].dt.strftime('%Y-%m')
orders[['order_id', 'order_date', 'order_month']].head()

,order_id,order_date,order_month
0,ORD-1001,2025-03-08,2025-03
2,ORD-1003,2025-03-22,2025-03
3,ORD-1004,2025-04-15,2025-04
4,ORD-1005,2025-05-01,2025-05
5,ORD-1006,2025-01-16,2025-01


## 12.8 Dealing with Impossible Values

In [23]:
orders[['quantity', 'unit_price']].describe()

,quantity,unit_price
count,188.000000,188.000000
mean,109.175532,249.859574
std,1028.266459,331.074147
min,1.000000,-259.000000
25%,1.000000,49.900000
50%,3.000000,159.000000
75%,4.000000,259.000000
max,9999.000000,1099.000000


In [24]:
q1 = orders['quantity'].quantile(0.25)
q3 = orders['quantity'].quantile(0.75)
upper_limit = q3 + 1.5 * (q3 - q1)
orders.loc[orders['quantity'] > upper_limit,
           ['order_id', 'product', 'quantity']]

,order_id,product,quantity
43,ORD-1043,Mouse,9999.0
162,ORD-1158,Mouse,9999.0


In [25]:
orders.loc[orders['unit_price'] <= 0,
           ['order_id', 'product', 'unit_price']]

,order_id,product,unit_price
13,ORD-1014,Webcam,-259.0
106,ORD-1102,USB Cable,-19.9


In [26]:
valid = (orders['quantity'] <= upper_limit) & (orders['unit_price'] > 0)
orders = orders[valid].copy()
orders['quantity'] = orders['quantity'].astype('int64')
orders['revenue'] = orders['quantity'] * orders['unit_price']
orders.shape

(184, 9)

## 12.9 Merging Customer Data

In [27]:
customers = pd.read_csv('customers.csv')
print(customers.shape)
print(customers['customer_id'].is_unique)

(50, 3)
True


In [28]:
merged = pd.merge(orders, customers, on='customer_id', how='left',
                  validate='many_to_one', indicator=True)
merged['_merge'].value_counts()

_merge
both          181
left_only       3
right_only      0
Name: count, dtype: int64

In [29]:
merged.loc[merged['_merge'] == 'left_only',
           ['order_id', 'customer_id', 'revenue']]

,order_id,customer_id,revenue
79,ORD-1085,C051,39.8
118,ORD-1126,C051,567.0
163,ORD-1174,C052,59.7


In [30]:
merged['segment'] = merged['segment'].fillna('Unknown')
orders_clean = merged.drop(columns=['_merge', 'customer_name'])
orders_clean.shape

(184, 10)

## 12.10 Validating the Clean Dataset

In [31]:
expected_cities = ['New York', 'Philadelphia', 'Los Angeles', 'Denver',
                   'San Francisco', 'Seattle', 'Unknown']
checks = pd.Series({
    'no missing values': orders_clean.notna().all().all(),
    'order_id unique': orders_clean['order_id'].is_unique,
    'ids like C001': orders_clean['customer_id'].str.contains(
        r'^C\d{3}$').all(),
    'quantity 1 to 5': orders_clean['quantity'].between(1, 5).all(),
    'price positive': (orders_clean['unit_price'] > 0).all(),
    'known cities': orders_clean['city'].isin(expected_cities).all(),
    'dates in H1 2025': orders_clean['order_date'].between(
        '2025-01-01', '2025-06-30').all(),
})
checks

no missing values    True
order_id unique      True
ids like C001        True
quantity 1 to 5      True
price positive       True
known cities         True
dates in H1 2025     True
dtype: bool

In [32]:
orders_clean.dtypes

order_id                  str
order_date     datetime64[us]
customer_id               str
product                   str
city                      str
quantity                int64
unit_price            float64
order_month               str
revenue               float64
segment                   str
dtype: object

## 12.11 Summarizing Sales

In [33]:
monthly = orders_clean.pivot_table(
    index='city', columns='order_month', values='revenue',
    aggfunc='sum', fill_value=0,
).round(0)
monthly

order_month,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06
city,,,,,,
Denver,1394.0,2200.0,339.0,906.0,60.0,0.0
Los Angeles,1652.0,5718.0,7826.0,914.0,7480.0,1103.0
New York,4921.0,10373.0,2508.0,15765.0,8737.0,11206.0
Philadelphia,4805.0,3493.0,2189.0,2537.0,6102.0,6538.0
San Francisco,6644.0,1005.0,1584.0,1731.0,1819.0,965.0
Seattle,2688.0,707.0,916.0,0.0,1116.0,824.0
Unknown,516.0,0.0,6758.0,5495.0,0.0,279.0


In [34]:
segment_summary = (
    orders_clean.groupby('segment')
    .agg(orders=('order_id', 'count'), revenue=('revenue', 'sum'))
    .sort_values('revenue', ascending=False)
)
segment_summary

,orders,revenue
segment,,
Consumer,142,112493.5
Business,39,28648.6
Unknown,3,666.5


In [35]:
pivot_total = monthly.sum().sum()
segment_total = segment_summary['revenue'].sum()
print(round(pivot_total), round(segment_total))

141813 141809


## 12.12 Exporting the Clean File

In [36]:
orders_clean.to_csv('orders_clean.csv', index=False)
check = pd.read_csv('orders_clean.csv', parse_dates=['order_date'])
print(check.shape)
print(check['order_date'].dtype)

(184, 10)
datetime64[us]
